# H16 · New data: do energy, climate, tax, banking and currency indicators explain profitability?

> **Hypothesis H16:** At least one of the new country indicators (taxes, energy and climate, currency, banking health, investment, labour and education) is related to company ROA beyond the development index.

| | |
|---|---|
| **Why we ask** | The 35 indicators used so far are mostly about wealth and governance. Companies are also shaped by energy costs, currency swings, a stressed banking system or high taxes. We added 13 further World Bank indicators to see if any of them carries information that development and institutions miss. |
| **Prediction** | Any association (no direction pre-agreed) between a new indicator and the country median ROA; the best of 12 must survive a multiple-testing correction. |
| **Data** | 13 new World Bank indicators (country averages 2019–2023, fetched by `src/fetch/extra_indicators.py`): income-tax share of revenue, CO₂ per person, renewable energy share, energy intensity, currency depreciation against the dollar, real interest rate, bank non-performing loans, investment (gross capital formation), government consumption, female labour participation, stock-market turnover, broadband use, secondary enrolment. Corporate tax *rates* have no free series in the World Bank data and are not included. |
| **Primary test (fixed before looking at the result)** | Spearman ρ between each indicator and the country median ROA (Yahoo, fiscal 2022–2025, countries with ≥ 10 firms), **Benjamini–Hochberg corrected across the 12 indicators with data for at least 15 countries** (real interest rates cover only 13 countries). The test statistic is the strongest indicator's ρ with its corrected q-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. Here the p-value carried to the scoreboard is the **q-value** of the strongest indicator (already corrected for the 12 looks); *robust share* is the share of seven firm samples in which that indicator keeps its sign. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The new indicators
Coverage and basic description. Country values are 2019–2023 means (at least 3 yearly values); *currency depreciation* is the average annual rise in local currency per US dollar (positive = the local currency lost value).

In [2]:
E = pd.read_parquet(ROOT / "data" / "flat" / "extra_country_year.parquet")
E = E[E.iso3.isin(dim.index)]
fxp = E[E.indicator == "fx_lcu_per_usd"].pivot(index="year", columns="iso3", values="value")
dep = (np.log(fxp).diff() * 100).loc[2019:2023].mean().rename("fx_depreciation")
w = E[(E.indicator != "fx_lcu_per_usd") & E.year.between(2019, 2023)].groupby(["iso3", "indicator"]).value.agg(lambda s: s.mean() if s.notna().sum() >= 3 else np.nan).unstack()
x = w.join(dep)
LABEL = {"taxes_on_income_share_of_revenue": "income taxes, % of revenue", "co2_per_capita": "CO₂ per person (t)", "renewable_share": "renewable energy share (%)", "energy_intensity": "energy intensity (MJ per $ GDP)",
         "fx_depreciation": "currency depreciation vs USD (% a year)", "real_interest_rate": "real interest rate (%)", "bank_npl_share": "bank non-performing loans (%)", "gross_capital_formation": "investment (% of GDP)",
         "gov_consumption": "government consumption (% of GDP)", "female_participation": "female labour participation (%)", "stock_turnover": "stock-market turnover (%)",
         "broadband_per_100": "broadband per 100 people", "secondary_enrolment": "secondary enrolment (%)"}
x = x[[c for c in LABEL if c in x.columns]]
cov = pd.DataFrame({"indicator": [LABEL[c] for c in x.columns], "countries with data": x.notna().sum().values, "mean": x.mean().values, "SD": x.std().values}, index=x.columns)
cov

,indicator,countries with data,mean,SD
taxes_on_income_share_of_revenue,"income taxes, % of revenue",31,47.228,16.437
co2_per_capita,CO₂ per person (t),34,7.053,3.570
renewable_share,renewable energy share (%),34,22.316,13.551
energy_intensity,energy intensity (MJ per $ GDP),34,3.510,1.293
fx_depreciation,currency depreciation vs USD (% a year),34,2.308,1.787
real_interest_rate,real interest rate (%),13,3.587,7.318
bank_npl_share,bank non-performing loans (%),32,2.757,3.262
gross_capital_formation,investment (% of GDP),34,24.521,5.542
gov_consumption,government consumption (% of GDP),34,18.778,4.382
female_participation,female labour participation (%),34,53.199,7.177


## 2 · Primary test: screen against profitability
Country median ROA (non-pooled: one number per country) against each new indicator, Spearman with bootstrap intervals, Benjamini–Hochberg across the 12 indicators that have at least 15 countries.

In [3]:
f5 = fy[fy.fiscal_year.between(2022, 2025)]
roa_c = f5.groupby("iso3").agg(roa=("roa_pp", "median"), n=("company_id", "nunique")).query("n >= @MIN_FIRMS")
sc = scan(x.loc[x.index.intersection(roa_c.index)], roa_c.roa.reindex(x.index.intersection(roa_c.index)), list(x.columns), min_n=15)
sc["indicator"] = sc.index.map(LABEL)
sc["dev_rho"] = [spearman(x[f].reindex(country["dev_idx"].index), country["dev_idx"])[0] for f in sc.index]            # overlap with the development index
sc["partial ρ (given development)"] = [partial_rho(x[f].reindex(roa_c.index), roa_c.roa, country["dev_idx"].reindex(roa_c.index)) for f in sc.index]
fig = px.bar(sc.reset_index(), x="rho", y="indicator", orientation="h", color=sc.q.values < 0.10, title="New indicators vs country median ROA (Spearman ρ; colour = BH q < 0.10)", labels={"color": "q < 0.10"})
fig.update_layout(height=470, yaxis_autorange="reversed"); fig.show()
top = sc.iloc[0]; topf = sc.index[0]
print(f"{len(sc)} indicators screened on {roa_c.shape[0]} countries. Strongest: {top.indicator} (ρ = {top.rho:+.2f}, p = {top.p:.4f}, BH q = {top.q:.3f}, n = {int(top.n)}); passing q < 0.10: {int((sc.q < 0.10).sum())}.")
sc[["indicator", "rho", "p", "q", "n", "dev_rho", "partial ρ (given development)"]].round(3)

12 indicators screened on 34 countries. Strongest: broadband per 100 people (ρ = -0.40, p = 0.0191, BH q = 0.177, n = 34); passing q < 0.10: 0.


,indicator,rho,p,q,n,dev_rho,partial ρ (given development)
feature,,,,,,,
broadband_per_100,broadband per 100 people,-0.400,0.019,0.177,34,0.569,-0.261
gov_consumption,government consumption (% of GDP),-0.374,0.030,0.177,34,0.470,-0.253
energy_intensity,energy intensity (MJ per $ GDP),0.309,0.076,0.302,34,-0.222,0.253
secondary_enrolment,secondary enrolment (%),-0.144,0.423,0.997,33,0.526,0.019
stock_turnover,stock-market turnover (%),0.139,0.548,0.997,21,0.104,0.211
female_participation,female labour participation (%),-0.114,0.523,0.997,34,0.699,0.195
co2_per_capita,CO₂ per person (t),-0.095,0.595,0.997,34,0.494,0.096
fx_depreciation,currency depreciation vs USD (% a year),0.035,0.844,0.997,34,-0.190,-0.034
renewable_share,renewable energy share (%),-0.030,0.866,0.997,34,-0.067,-0.057


### The leading indicator in a plot
One point per country.

In [4]:
d = pd.DataFrame({"x": x[topf], "roa": roa_c.roa}).dropna().join(dim.name)
fig = px.scatter(d.reset_index(), x="x", y="roa", text="iso3", hover_name="name", title=f"{top.indicator} vs median ROA (ρ = {top.rho:+.2f})", labels={"x": top.indicator, "roa": "median ROA (pp)"}, height=460)
b, a = np.polyfit(d.x, d.roa, 1); xs = np.array([d.x.min(), d.x.max()]); fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False))
fig.update_traces(textposition="top center", selector=dict(mode="markers+text")); fig.show()

## 3 · Do the new indicators add anything for prediction?
Leave-one-country-out prediction of the country median ROA (ridge regression, all new indicators standardised, missing values at the average) against the development index alone and against a constant. The gain in mean absolute error (MAE) is what matters.

In [5]:
from sklearn.linear_model import Ridge
yv = roa_c.roa; idx = yv.index
Xnew = x.reindex(idx); Xnew = ((Xnew - Xnew.mean()) / Xnew.std()).fillna(0)
Xdev = country["dev_idx"].reindex(idx).to_frame("dev")
models = {"constant (median)": None, "development index only": Xdev, "new indicators only": Xnew, "development + new indicators": pd.concat([Xdev, Xnew], axis=1)}
res = {}
for lab, X in models.items():
    err = []
    for i in idx:
        tr = idx.drop(i)
        pred = yv[tr].median() if X is None else Ridge(alpha=5.0).fit(X.loc[tr], yv[tr]).predict(X.loc[[i]])[0]
        err.append(abs(yv[i] - pred))
    res[lab] = np.mean(err)
loco = pd.Series(res, name="MAE (pp ROA)").to_frame(); loco["gain vs constant"] = 1 - loco["MAE (pp ROA)"] / loco.loc["constant (median)", "MAE (pp ROA)"]
loco

,MAE (pp ROA),gain vs constant
constant (median),0.914,0.000
development index only,0.910,0.005
new indicators only,1.100,-0.203
development + new indicators,1.123,-0.228


### Firm level: the leading indicator with company controls
The strongest indicator, standardised across countries, as a country variable in the usual company-level model (sector, size, year; wild cluster bootstrap).

In [6]:
f5 = fy.copy(); z = (x[topf] - x[topf].mean()) / x[topf].std(); f5["lead_z"] = f5.iso3.map(z)
firm = wild_cluster("lead_z", f5, outcome="roa_pp", B=1999)
print(f"{top.indicator}: +1 SD → {firm['coef']:+.2f} pp ROA (95% CI {firm['ci_low']:+.2f} to {firm['ci_high']:+.2f}), wild p = {firm['p_wild']:.3f}, {firm['clusters']} countries.")
f5["dev_static"] = f5.iso3.map(country["dev_idx"])
both = wild_cluster("lead_z", f5, outcome="roa_pp", extra="dev_static", B=1999)
print(f"…controlling for the development index: {both['coef']:+.2f} pp (95% CI {both['ci_low']:+.2f} to {both['ci_high']:+.2f}), wild p = {both['p_wild']:.3f}.")

broadband per 100 people: +1 SD → -0.97 pp ROA (95% CI -1.44 to -0.49), wild p = 0.007, 34 countries.


…controlling for the development index: -0.36 pp (95% CI -0.87 to +0.16), wild p = 0.191.


## 4 · Robustness
The leading indicator's country-level ρ with the median ROA recomputed in the seven usual firm samples.

In [7]:
rows = []
for nm, flt in SAMPLES.items():
    g = flt(fy).groupby("iso3").agg(roa=("roa_pp", "median"), n=("company_id", "nunique")).query("n >= @MIN_FIRMS")
    r_, p_, n_ = spearman(x[topf].reindex(g.index), g.roa, 12)
    rows.append({"sample": nm, "ρ": r_, "p": p_, "countries": n_, "same sign": np.sign(r_) == np.sign(top.rho)})
rob = pd.DataFrame(rows).set_index("sample"); rob_share = float(rob["same sign"].mean())
fig = px.bar(rob.reset_index(), x="ρ", y="sample", orientation="h", title=f"{top.indicator}: ρ with median ROA in different samples"); fig.update_layout(height=380, yaxis_autorange="reversed"); fig.show()
print(f"Same sign in {rob_share:.0%} of samples.")
rob

Same sign in 100% of samples.


,ρ,p,countries,same sign
sample,,,,
all firms,-0.406,0.017,34,True
excluding financials & real estate,-0.308,0.081,33,True
excluding small-sample countries,-0.419,0.024,29,True
"excluding USA, China, Japan",-0.493,0.005,31,True
fiscal years 2023 and later,-0.398,0.022,33,True
EU countries only,-0.218,0.342,21,True
non-EU countries only,-0.440,0.133,13,True


## 5 · Replication on ESEF
The same indicator against the country median ROA of the **EU** companies in ESEF (about 19 countries; partly the same companies as Yahoo, see H15).

In [8]:
fe = prep.load_firms(con, country, source="ESEF"); fe = fe[fe.fiscal_year.between(2022, 2025)]
ce = fe.groupby("iso3").agg(roa=("roa_pp", "median"), n=("company_id", "nunique")).query("n >= @MIN_FIRMS")
rho_e, p_e, n_e = spearman(x[topf].reindex(ce.index), ce.roa, 8)
rep_same = bool(np.sign(rho_e) == np.sign(top.rho))
print(f"ESEF country medians ({n_e} EU countries): ρ = {rho_e:+.2f} (p = {p_e:.3f}); same sign as the main result: {rep_same}.")

ESEF country medians (17 EU countries): ρ = -0.18 (p = 0.480); same sign as the main result: True.


## 6 · Verdict

In [9]:
v = utils.verdict(top.rho, float(top.q), 0, rob_share, int(top.n))
lines = [
 f"Primary: strongest of {len(sc)} new indicators is '{top.indicator}': ρ = {top.rho:+.2f} with median ROA (raw p = {top.p:.4f}, BH q = {top.q:.3f}, n = {int(top.n)} countries); {int((sc.q < 0.10).sum())} indicators pass q < 0.10.",
 f"Given the development index the partial ρ is {top['partial ρ (given development)']:+.2f}; firm-level coefficient {firm['coef']:+.2f} pp per SD (wild p = {firm['p_wild']:.3f}), {both['coef']:+.2f} pp with development controlled (p = {both['p_wild']:.3f}).",
 f"Robustness: same sign in {rob_share:.0%} of seven samples. ESEF (EU only) ρ = {rho_e:+.2f} (p = {p_e:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Prediction (leave-one-country-out MAE): constant {loco.iloc[0, 0]:.2f}, development {loco.iloc[1, 0]:.2f}, new indicators {loco.iloc[2, 0]:.2f}, both {loco.iloc[3, 0]:.2f} pp.",
 f"VERDICT H16: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H16", id="H16", title="New indicators: energy, climate, tax, banking, currency", statement="At least one new country indicator is related to company ROA beyond development.",
                  expected_sign=0, effect=float(top.rho), ci_low=None, ci_high=None, effect_unit=f"Spearman ρ, {top.indicator} vs country median ROA (strongest of {len(sc)})",
                  p_primary=float(top.q), p_method="Spearman p, BH-corrected over the new indicators", n=int(top.n), clusters=int(top.n), robust_share=rob_share,
                  replication_coef=float(rho_e), replication_p=float(p_e), replication_same_sign=rep_same, verdict=v,
                  notebook="1.16_new_country_indicators.ipynb", notes=lines)

Primary: strongest of 12 new indicators is 'broadband per 100 people': ρ = -0.40 with median ROA (raw p = 0.0191, BH q = 0.177, n = 34 countries); 0 indicators pass q < 0.10.
Given the development index the partial ρ is -0.26; firm-level coefficient -0.97 pp per SD (wild p = 0.007), -0.36 pp with development controlled (p = 0.191).
Robustness: same sign in 100% of seven samples. ESEF (EU only) ρ = -0.18 (p = 0.480), same direction.
Prediction (leave-one-country-out MAE): constant 0.91, development 0.91, new indicators 1.10, both 1.12 pp.
VERDICT H16: NOT SUPPORTED


### Limits
- Twelve indicators looked at once: the BH correction handles that, but the *choice* of these 13 (and not others) was ours, and some are strongly related to development (see the `dev_rho` column).
- 2019–2023 averages smooth over very different years (pandemic, inflation shock); real interest rates and stock turnover are missing for many countries.
- A country-level ρ with 20–34 countries is a weak basis for any claim; the firm-level and prediction checks are there to show that.
- Association, not causation.

In [10]:
con.close()